# Notebook 4: Advanced MLflow Tracking

**Learning Objectives:**
- Use tags for better organization
- Log complex artifacts (plots, confusion matrices)
- Implement nested runs for experiments
- Use MLflow autologging
- Track custom metrics

**Dataset:** Iris (multi-class classification)

In [1]:
import mlflow
from mlflow.models import infer_signature
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

## Step 1: Prepare Data

In [2]:
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dataset loaded: {X.shape}")
print(f"Classes: {iris.target_names}")

Dataset loaded: (150, 4)
Classes: ['setosa' 'versicolor' 'virginica']


## Feature 1: Using Tags for Organization

In [3]:
mlflow.set_experiment("04_Advanced_Tracking")

with mlflow.start_run(run_name="Tagged_Experiment"):
    
    # Set tags for better organization
    mlflow.set_tags({
        "team": "data-science",
        "project": "iris-classification",
        "version": "v1.0",
        "environment": "development",
        "algorithm": "random-forest",
        "author": "student-demo"
    })
    
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Log parameters
    mlflow.log_params({
        "n_estimators": 100,
        "random_state": 42,
        "dataset": "iris",
        "n_features": X.shape[1]
    })
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)
    
    print(f" Experiment tagged and logged (accuracy: {accuracy:.4f})")

print("\n Check MLflow UI - you can now filter runs by tags!")

2026/02/16 13:09:25 INFO mlflow.tracking.fluent: Experiment with name '04_Advanced_Tracking' does not exist. Creating a new experiment.


 Experiment tagged and logged (accuracy: 0.9000)

 Check MLflow UI - you can now filter runs by tags!


## Feature 2: Logging Complex Artifacts

In [9]:
with mlflow.start_run(run_name="Artifacts_Demo"):
    
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=iris.target_names, 
                yticklabels=iris.target_names)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png')
    mlflow.log_artifact('confusion_matrix.png')
    plt.close()
    
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 6))
    plt.title('Feature Importances')
    plt.bar(range(X.shape[1]), importances[indices], color='lightblue')
    plt.xticks(range(X.shape[1]), [iris.feature_names[i] for i in indices], rotation=45, ha='right')
    plt.ylabel('Importance')
    plt.tight_layout()
    plt.savefig('feature_importance.png')
    mlflow.log_artifact('feature_importance.png')
    plt.close()
    
    report = classification_report(y_test, y_pred, target_names=iris.target_names, output_dict=True)
    with open('classification_report.json', 'w') as f:
        json.dump(report, f, indent=2)
    mlflow.log_artifact('classification_report.json')
    
    predictions_df = pd.DataFrame({
        'true_label': y_test,
        'predicted_label': y_pred,
        'true_class': [iris.target_names[i] for i in y_test],
        'predicted_class': [iris.target_names[i] for i in y_pred]
    })
    predictions_df.to_csv('predictions.csv', index=False)
    mlflow.log_artifact('predictions.csv')
    
    print(" Logged multiple artifacts:")
    print("   - Confusion matrix (PNG)")
    print("   - Feature importance (PNG)")
    print("   - Classification report (JSON)")
    print("   - Predictions (CSV)")

 Logged multiple artifacts:
   - Confusion matrix (PNG)
   - Feature importance (PNG)
   - Classification report (JSON)
   - Predictions (CSV)


## Feature 3: Model Signature and Input Example

In [5]:
with mlflow.start_run(run_name="Model_With_Signature"):
    
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    signature = infer_signature(X_train, model.predict(X_train))
    
    input_example = X_train[:3]
    
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=signature,
        input_example=input_example
    )
    
    y_pred = model.predict(X_test)
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    
    print(" Model logged with:")
    print("   - Signature (input/output schema)")
    print("   - Input example (for documentation)")
    print("\n This helps with model deployment and validation!")

2026/02/16 13:09:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


 Model logged with:
   - Signature (input/output schema)
   - Input example (for documentation)

 This helps with model deployment and validation!


## Feature 4: Nested Runs (Parent-Child Experiments)

In [6]:
with mlflow.start_run(run_name="Cross_Validation_Experiment") as parent_run:
    
    mlflow.set_tag("experiment_type", "cross-validation")
    
    all_scores = []
    
    for fold in range(5):
        with mlflow.start_run(run_name=f"Fold_{fold+1}", nested=True):
            
            X_fold_train, X_fold_test, y_fold_train, y_fold_test = train_test_split(
                X, y, test_size=0.2, random_state=fold, stratify=y
            )
            
            model = RandomForestClassifier(n_estimators=50, random_state=42)
            model.fit(X_fold_train, y_fold_train)
            
            y_pred = model.predict(X_fold_test)
            accuracy = accuracy_score(y_fold_test, y_pred)
            all_scores.append(accuracy)
            
            mlflow.log_param("fold", fold + 1)
            mlflow.log_metric("fold_accuracy", accuracy)
            
            print(f"Fold {fold+1}: accuracy={accuracy:.4f}")
    
    mlflow.log_metric("cv_mean_accuracy", np.mean(all_scores))
    mlflow.log_metric("cv_std_accuracy", np.std(all_scores))
    mlflow.log_param("n_folds", 5)
    
    print(f"\n Cross-validation completed:")
    print(f"   Mean accuracy: {np.mean(all_scores):.4f} ± {np.std(all_scores):.4f}")
    print("\n Check MLflow UI - you'll see nested runs under the parent!")

Fold 1: accuracy=1.0000
Fold 2: accuracy=0.9667
Fold 3: accuracy=0.9667
Fold 4: accuracy=0.9000
Fold 5: accuracy=1.0000

 Cross-validation completed:
   Mean accuracy: 0.9667 ± 0.0365

 Check MLflow UI - you'll see nested runs under the parent!


## Feature 5: MLflow Autologging

In [7]:
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="Autolog_Demo"):
    
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    print("Model trained with autologging!")
    print("\nMLflow automatically logged:")
    print("   - All model parameters")
    print("   - Training metrics")
    print("   - Model artifact")
    print("   - Model signature")
    print("\n Check MLflow UI to see all auto-logged content!")

mlflow.sklearn.autolog(disable=True)
print("\n  Autologging disabled")

Model trained with autologging!

MLflow automatically logged:
   - All model parameters
   - Training metrics
   - Model artifact
   - Model signature

 Check MLflow UI to see all auto-logged content!

  Autologging disabled


## Feature 6: Custom Metrics and Parameters

In [8]:
with mlflow.start_run(run_name="Custom_Metrics"):
    
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Log standard metrics
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    
    # Custom metrics
    mlflow.log_metric("test_samples", len(y_test))
    mlflow.log_metric("train_samples", len(y_train))
    mlflow.log_metric("n_trees", model.n_estimators)
    mlflow.log_metric("avg_tree_depth", np.mean([tree.get_depth() for tree in model.estimators_]))
    mlflow.log_metric("max_tree_depth", np.max([tree.get_depth() for tree in model.estimators_]))
    
    # Custom parameters (metadata)
    mlflow.log_param("dataset_name", "iris")
    mlflow.log_param("test_split_ratio", 0.2)
    mlflow.log_param("stratified", True)
    
    data_info = {
        "n_samples": len(X),
        "n_features": X.shape[1],
        "n_classes": len(np.unique(y))
    }
    mlflow.log_dict(data_info, "data_info.json")
    
    print(" Logged custom metrics and parameters:")
    print("   - Tree statistics (depth, count)")
    print("   - Dataset metadata")
    print("   - Version information")

 Logged custom metrics and parameters:
   - Tree statistics (depth, count)
   - Dataset metadata
   - Version information


##  Key Takeaways

1. **Tags** - Organize experiments by team, project, version, etc.
2. **Artifacts** - Log visualizations, reports, data files
3. **Signatures** - Document model input/output schemas
4. **Nested Runs** - Organize related experiments hierarchically
5. **Autologging** - Automatic logging for supported frameworks
6. **Custom Metrics** - Track domain-specific measurements

##  Exercise

1. Explore all runs in MLflow UI
2. Filter runs by tags
3. View artifacts in the artifact viewer
4. Compare nested runs to parent run
5. Check what autologging captured automatically